In [ ]:
import os
import json
import math
import random
import shutil
import gc
from typing import Dict, Any, List, Optional, Tuple, Literal

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    matthews_corrcoef
)
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns
import wandb

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 42
set_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Seed: {seed} | Device: {device}")

Seed: 42 | Device: cuda:0


In [ ]:
wandb_project_name = "model_settransformer_test"
experiment_name = 'model_settransformer_test'
checkpoint_dir = '/path/to/experiments/checkpoints'
MODEL_DIM=8192

In [ ]:
var = pd.read_feather('/path/to/data/embedding.feather')
label = pd.read_feather('/path/to/data/label.feather')


In [10]:
if var.columns[0] != 'vcf_iid':
    var.columns.values[0] = 'vcf_iid'

In [12]:
a = np.stack(var['mut_emb'].values)

In [ ]:
class SetTransformer(nn.Module):
    def __init__(self,
                 input_dim: int,
                 num_heads: int,
                 num_encoder_layers: int,
                 activation: Literal['relu', 'gelu'],
                 dropout: float = 0.1):
        super(SetTransformer, self).__init__()
        self.input_dim = input_dim
        self.attn_scores = []


        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            dim_feedforward=input_dim * 4,
            nhead=num_heads,
            activation=activation,
            dropout=dropout,  
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_encoder_layers,
            enable_nested_tensor=False
        )


    def forward(self,
                emb: torch.Tensor,
                pad_mask: Optional[torch.Tensor]) -> torch.Tensor:

        return self.transformer(emb, src_key_padding_mask=pad_mask)


class PMA(nn.Module):
    def __init__(self,
                 dim_Q: int,
                 dim_K: int,
                 dim_V: int,
                 num_heads: int,
                 dropout: float = 0.1,
                 initial_temperature: float = 0.1):
        super(PMA, self).__init__()
        self.dim_V = dim_V
        self.num_heads = num_heads
        
        self.fc_q = nn.Linear(dim_Q, dim_V)
        self.fc_k = nn.Linear(dim_K, dim_V)
        self.fc_v = nn.Linear(dim_K, dim_V)
        
        self.fc_o = nn.Linear(dim_V, dim_V)
        
        self.ln1 = nn.LayerNorm(dim_V)
        self.ln2 = nn.LayerNorm(dim_V)
        
        self.ffn = nn.Sequential(
            nn.Linear(dim_V, dim_V * 4),
            nn.GELU(), 
            nn.Linear(dim_V * 4, dim_V),
            nn.Dropout(dropout)
        )
        
        self.dropout = nn.Dropout(dropout)
        self.attn_scores = None
        self.temperature = nn.Parameter(torch.tensor([initial_temperature], dtype=torch.float32))


    def forward(self,
                Q: torch.Tensor,
                K: torch.Tensor,
                mask: Optional[torch.Tensor]) -> torch.Tensor:
        
        batch_size, num_keys = K.size(0), K.size(1)
        
        Q_proj = self.fc_q(Q) 
        K_proj = self.fc_k(K)
        V_proj = self.fc_v(K)

        dim_split = self.dim_V // self.num_heads
        Q_ = torch.cat(Q_proj.split(dim_split, 2), 0) 
        K_ = torch.cat(K_proj.split(dim_split, 2), 0) 
        V_ = torch.cat(V_proj.split(dim_split, 2), 0) 

        scale = math.sqrt(dim_split) * self.temperature
        A = Q_.bmm(K_.transpose(1, 2)) / scale
        
        if mask is not None:
            mask_ex = mask.unsqueeze(1).unsqueeze(2) 
            mask_ex = mask_ex.repeat(1, self.num_heads, 1, 1) 
            mask_ex = mask_ex.view(batch_size * self.num_heads, 1, num_keys) 
            
            A.masked_fill_(mask_ex, float("-inf"))
            
        A = torch.softmax(A, 2)
        
        with torch.no_grad():
            self.attn_scores = A.view(self.num_heads, batch_size, 1, num_keys).mean(dim=0).squeeze(1)

        A = self.dropout(A) 

        O_attn = torch.cat((A.bmm(V_)).split(Q.size(0), 0), 2)
        O_attn = self.fc_o(O_attn) 
        

        O = self.ln1(Q + self.dropout(O_attn))
        
        ffn_out = self.ffn(O)
        O = self.ln2(O + ffn_out)
        return O


class Pooler(nn.Module):
    def __init__(self,
                 input_dim: int,
                 normalize: Optional[Literal["l2"]]=None,
                 num_heads: Optional[int]=None,
                 dropout: float = 0.1,
                 initial_temperature: float = 0.1):
        super(Pooler, self).__init__()

        assert num_heads is not None
            
        self.seed_vector = nn.Parameter(torch.Tensor(1, 1, input_dim))
        nn.init.xavier_uniform_(self.seed_vector)
        self.pma = PMA(input_dim, input_dim, input_dim, num_heads, dropout=dropout,initial_temperature=initial_temperature)

        self.normalize = normalize
    

    def forward(self,
                   embs: torch.Tensor,
                   mask: Optional[torch.Tensor]) -> torch.Tensor:
        if self.normalize == "l2":
            embs = embs / (torch.norm(embs, p=2, dim=-1, keepdim=True) + 1e-8)

        return self.pma(self.seed_vector.repeat(embs.size(0), 1, 1), embs, mask).squeeze(1)
    

class SLP(nn.Module):
    def __init__(self,
                 input_dim: int,
                 output_dim: int,
                 dropout: float = 0.1):
        super(SLP, self).__init__()

        self.fc = nn.Linear(input_dim, output_dim)
        self.dropout = nn.Dropout(dropout)


    def forward(self,
                embs: torch.tensor) -> torch.tensor:
        embs = self.dropout(embs)  
        return self.fc(embs)


class MLP(nn.Module):
    def __init__(self,
                 input_dim: int,
                 hidden_dims: list,
                 activation: Optional[Literal["relu", "leakyrelu", "tanh", "sigmoid"]],
                 output_dim: int,
                 dropout: float = 0.1):
        super(MLP, self).__init__()
        act_funcs = {
            "relu": nn.ReLU,
            "leakyrelu": nn.LeakyReLU,
            "tanh": nn.Tanh,
            "sigmoid": nn.Sigmoid
        }

        dims = [input_dim]
        dims.extend(hidden_dims)
        layers = []

        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if activation != "none":
                layers.append(act_funcs[activation]())
            layers.append(nn.Dropout(dropout)) 
        layers.append(nn.Linear(dims[-1], output_dim))

        self.fwd_layers = nn.Sequential(*layers)


    def forward(self,
                embs: torch.Tensor) -> torch.Tensor:

        return self.fwd_layers(embs)



class ASDPredictor(nn.Module):
    def __init__(self,
                 configs_of_modules: dict,
                 device: str):
        
        super().__init__()
        self.device = torch.device(device)
        self.attn_score = None

        self.layers = nn.ModuleList([
            SetTransformer(**configs_of_modules["set_transformer"]).to(device),
            Pooler(**configs_of_modules["pooling"]).to(device),
            MLP(**configs_of_modules["linear"]).to(device)
        ])

    def forward(self,
                embs: torch.Tensor, pad_mask: torch.Tensor):

        pre_pma_embs = None
        post_pma_embs = None

        x = embs

        for layer in self.layers:
            if isinstance(layer, SetTransformer):
                x = layer(x, pad_mask)
                pre_pma_embs = x   

            elif isinstance(layer, Pooler):
                x = layer(x, pad_mask)
                post_pma_embs = x  
                self.attn_score = layer.pma.attn_scores

            else:
                logits = layer(x)

        return logits, pre_pma_embs, post_pma_embs


In [ ]:
def build_split_tensors_from_df(
    df_emb: pd.DataFrame,
    df_label: pd.DataFrame,
    emb_col: str = "emb_patho",
    id_col: str = "vcf_iid",
    label_col: str = "label",
    split_col: str = "split",
    label_map: Dict[str, float] = None,
    make_global_maxlen: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:

    if label_map is None:
        label_map = {"Control": 0.0, "Autism": 1.0, "ASD": 1.0, "case": 1.0, "ctrl": 0.0}

    for c in [id_col, emb_col]:
        if c not in df_emb.columns:
            raise KeyError(f"df_emb missing column: {c}")
    for c in [id_col, label_col, split_col]:
        if c not in df_label.columns:
            raise KeyError(f"df_label missing column: {c}")

    lab = df_label[[id_col, label_col, split_col]].copy()
    lab[id_col] = lab[id_col].astype(str)

    def _to01(x):
        if isinstance(x, str):
            x2 = x.strip()
            if x2 in label_map:
                return float(label_map[x2])
            x2l = x2.lower()
            if x2l in label_map:
                return float(label_map[x2l])
            try:
                return float(x2l)
            except:
                return 0.0
        return float(x)

    lab["y"] = lab[label_col].apply(_to01).astype(np.float32)

    split_to_ids = {
        s: lab.loc[lab[split_col] == s, id_col].astype(str).tolist()
        for s in ["train", "val", "test"]
    }

    embdf = df_emb[[id_col, emb_col]].copy()
    embdf[id_col] = embdf[id_col].astype(str)

    groups = embdf.groupby(id_col, sort=False)[emb_col].apply(list).to_dict()

    def _build_one_split(sample_ids: List[str]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        X_list = []
        y_list = []

        y_lookup = dict(zip(lab[id_col].values, lab["y"].values))

        for sid in sample_ids:
            if sid not in groups:
                continue
            seq = groups[sid]  
            arr = np.stack([np.asarray(v, dtype=np.float32) for v in seq], axis=0)
            X_list.append(arr)
            y_list.append(y_lookup[sid])

        if len(X_list) == 0:
            raise ValueError("No samples found after matching labels with embeddings. Check vcf_iid consistency.")

        lengths = [x.shape[0] for x in X_list]
        D = X_list[0].shape[1]
        max_len = max(lengths)

        X = np.zeros((len(X_list), max_len, D), dtype=np.float32)
        pad = np.ones((len(X_list), max_len), dtype=np.bool_)  
        y = np.asarray(y_list, dtype=np.float32)

        for i, x in enumerate(X_list):
            L = x.shape[0]
            X[i, :L, :] = x
            pad[i, :L] = False

        return torch.from_numpy(X), torch.from_numpy(y), torch.from_numpy(pad)

    train_X, train_y, train_pad = _build_one_split(split_to_ids["train"])
    val_X,   val_y,   val_pad   = _build_one_split(split_to_ids["val"])
    test_X,  test_y,  test_pad  = _build_one_split(split_to_ids["test"])

    if make_global_maxlen:
        global_max = max(train_X.size(1), val_X.size(1), test_X.size(1))

        def _pad_to(tX, tpad):
            B, L, D = tX.shape
            if L == global_max:
                return tX, tpad
            newX = torch.zeros((B, global_max, D), dtype=tX.dtype)
            newpad = torch.ones((B, global_max), dtype=torch.bool)
            newX[:, :L, :] = tX
            newpad[:, :L] = tpad
            return newX, newpad

        train_X, train_pad = _pad_to(train_X, train_pad)
        val_X,   val_pad   = _pad_to(val_X, val_pad)
        test_X,  test_pad  = _pad_to(test_X, test_pad)

    return train_X, val_X, test_X, train_y, val_y, test_y, train_pad, val_pad, test_pad

In [ ]:
def prepare_dl(datasets: List[torch.Tensor],
               labels: List[torch.Tensor],
               pad_masks: List[torch.Tensor],
               batch_size: int,
               dl_seed: int,
               device: str) -> Tuple[DataLoader, DataLoader, DataLoader]:


    trainset, valset, testset = datasets
    train_lbl, val_lbl, test_lbl = labels
    train_mask, val_mask, test_mask = pad_masks

    train_td = TensorDataset(trainset, train_lbl, train_mask)
    val_td = TensorDataset(valset, val_lbl, val_mask)
    test_td = TensorDataset(testset, test_lbl, test_mask)

    g = torch.Generator()
    g.manual_seed(dl_seed)

    train_dl = DataLoader(train_td, batch_size=batch_size, shuffle=True, generator=g)
    val_dl = DataLoader(val_td, batch_size=batch_size, shuffle=False)
    test_dl = DataLoader(test_td, batch_size=batch_size, shuffle=False)

    return train_dl, val_dl, test_dl

In [ ]:
batch_size = 64

train_X, val_X, test_X, train_y, val_y, test_y, train_pad, val_pad, test_pad = build_split_tensors_from_df(
    df_emb=var,
    df_label=label,
    emb_col="mut_emb",
    id_col="vcf_iid",
    label_col="label",
    split_col="split",
    make_global_maxlen=False,
)

train_dl, val_dl, test_dl = prepare_dl(
    datasets=[train_X, val_X, test_X],
    labels=[train_y, val_y, test_y],
    pad_masks=[train_pad, val_pad, test_pad],
    batch_size=batch_size,
    dl_seed=seed,
    device=device
)

X, y, m = next(iter(train_dl))
print(X.shape, y.shape, m.shape, X.dtype, y.dtype, m.dtype)

torch.Size([64, 124, 8192]) torch.Size([64]) torch.Size([64, 124]) torch.float32 torch.float32 torch.bool


In [ ]:
class VICRegLoss(nn.Module):
    def __init__(self, sim_coeff=25.0, std_coeff=25.0, cov_coeff=1.0):
        super().__init__()
        self.sim_coeff = sim_coeff
        self.std_coeff = std_coeff
        self.cov_coeff = cov_coeff

    def off_diagonal(self, x):
        n, m = x.shape
        assert n == m
        return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()

    def forward(self, x, y):
        batch_size = x.shape[0]
        num_features = x.shape[1]

        repr_loss = F.mse_loss(x, y)

        std_x = torch.sqrt(x.var(dim=0) + 1e-04)
        std_y = torch.sqrt(y.var(dim=0) + 1e-04)
        std_loss = torch.mean(F.relu(1 - std_x)) + torch.mean(F.relu(1 - std_y))

        x = x - x.mean(dim=0)
        y = y - y.mean(dim=0)
        cov_x = (x.T @ x) / (batch_size - 1)
        cov_y = (y.T @ y) / (batch_size - 1)
        cov_loss = self.off_diagonal(cov_x).pow_(2).sum().div(num_features) + self.off_diagonal(cov_y).pow_(2).sum().div(num_features)

        return self.sim_coeff * repr_loss + self.std_coeff * std_loss + self.cov_coeff * cov_loss

class Projector(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=2048, output_dim=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim), 
        )

    def forward(self, x):
        if x.dim() > 2:
            x = x.squeeze()
        return self.net(x)

In [23]:
class WeightedBCEWithLogitsLoss(nn.Module):
    def __init__(self, w_pos=1.0, w_neg=1.0):
        super().__init__()
        self.register_buffer("w_pos", torch.tensor(w_pos))
        self.register_buffer("w_neg", torch.tensor(w_neg))

    def forward(self, logits, labels):
        loss_raw = F.binary_cross_entropy_with_logits(
            logits, labels, reduction="none"
        )
        sample_weight = labels * self.w_pos + (1.0 - labels) * self.w_neg
        return (loss_raw * sample_weight).mean()


In [ ]:
y_np = train_y.detach().cpu().numpy().astype(int).reshape(-1)
pos_num = int((y_np == 1).sum())
neg_num = int((y_np == 0).sum())
N = pos_num + neg_num

w_pos = N / (2.0 * pos_num) if pos_num > 0 else 1.0
w_neg = N / (2.0 * neg_num) if neg_num > 0 else 1.0

print("pos_num, neg_num:", pos_num, neg_num, "| w_pos, w_neg:", w_pos, w_neg)

pos_num, neg_num: 8299 3934 | w_pos, w_neg: 0.7370165080130137 1.5547788510421963


In [ ]:
def attention_entropy(attn: torch.Tensor, pad_mask: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    if attn is None:
        return torch.tensor(0.0, device=pad_mask.device)

    p = attn.masked_fill(pad_mask, 0.0)

    Z = p.sum(dim=-1, keepdim=True).clamp(min=eps)
    p = p / Z

    p_safe = p.clamp(min=eps)
    ent = -(p * torch.log(p_safe)).sum(dim=-1) 
    return ent.mean()


def get_attn_entropy_from_model(model, pad_mask: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    attn = getattr(model, "attn_score", None)
    if isinstance(attn, torch.Tensor):
        return attention_entropy(attn, pad_mask, eps=eps)
    return torch.tensor(0.0, device=pad_mask.device)

def calculate_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0


def compute_metrics(y_true, y_pred, y_prob):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    y_prob = np.array(y_prob).flatten()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "auroc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0,
        "auprc": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": calculate_specificity(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }

def train_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    epoch,
    gradient_clip=1.0,
    debug=False,
    attn_entropy_lambda: float = 0.0,
    attn_entropy_mode: str = "min",
    projector=None,
    vicreg_criterion=None,
    vicreg_lambda: float = 0.0
):
    model.train()
    
    if projector is not None:
        projector.train()

    total_loss = 0.0
    total_task = 0.0
    total_ent = 0.0
    total_vicreg = 0.0  

    all_preds, all_labels, all_probs = [], [], []

    for step, (X, y, pad_mask) in enumerate(loader):
        X = X.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.float32).view(-1, 1)
        pad_mask = pad_mask.to(device)   
        optimizer.zero_grad()

        logits, pre_pma_1, post_pma_1 = model(X, pad_mask)
        
        if isinstance(logits, (tuple, list)): logits = logits[0]
        if logits.dim() == 1: logits = logits.view(-1, 1)
        logits = logits.float()

        vicreg_loss = torch.tensor(0.0, device=device)
        
        if vicreg_lambda > 0 and projector is not None and vicreg_criterion is not None:
            _, _, post_pma_2 = model(X, pad_mask)
            z1 = projector(post_pma_1)
            z2 = projector(post_pma_2)

            vicreg_loss = vicreg_criterion(z1, z2)


        task_loss = criterion(logits, y)
        ent = get_attn_entropy_from_model(model, pad_mask)
        loss = task_loss
        
        if attn_entropy_lambda > 0:
            if attn_entropy_mode == "min":
                loss += attn_entropy_lambda * ent
            elif attn_entropy_mode == "max":
                loss -= attn_entropy_lambda * ent
            else:
                raise ValueError(f"Unknown attn_entropy_mode: {attn_entropy_mode}")
        
        if vicreg_lambda > 0:
            loss += vicreg_lambda * vicreg_loss
        loss.backward()

        if gradient_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            if projector is not None:
                torch.nn.utils.clip_grad_norm_(projector.parameters(), gradient_clip)

        optimizer.step()

        total_loss += float(loss.item())
        total_task += float(task_loss.item())
        total_ent  += float(ent.item())
        total_vicreg += float(vicreg_loss.item()) 

        probs = torch.sigmoid(logits).detach()
        preds = (probs > 0.5).float().detach()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.detach().cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / max(len(loader), 1)
    metrics = compute_metrics(all_labels, all_preds, all_probs)
    
    metrics["task_loss"] = total_task / max(len(loader), 1)
    metrics["attn_entropy"] = total_ent / max(len(loader), 1)
    metrics["vicreg_loss"] = total_vicreg / max(len(loader), 1) 

    return avg_loss, metrics


@torch.no_grad()
def validate(
    model, loader, criterion, device,
    attn_entropy_lambda: float = 0.0,
    attn_entropy_mode: str = "min",
):
    model.eval()
    total_loss = 0.0
    total_task = 0.0
    total_ent = 0.0

    all_preds, all_labels, all_probs = [], [], []

    for X, y, pad_mask in loader:
        X = X.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.float32).view(-1, 1)
        pad_mask = pad_mask.to(device)

        logits,_, _ = model(X, pad_mask)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]
        if logits.dim() == 1:
            logits = logits.view(-1, 1)
        logits = logits.float()

        task_loss = criterion(logits, y)
        ent = get_attn_entropy_from_model(model, pad_mask)

        if attn_entropy_lambda > 0:
            if attn_entropy_mode == "min":
                loss = task_loss + attn_entropy_lambda * ent
            elif attn_entropy_mode == "max":
                loss = task_loss - attn_entropy_lambda * ent
            else:
                raise ValueError(f"Unknown attn_entropy_mode: {attn_entropy_mode}")
        else:
            loss = task_loss

        total_loss += float(loss.item())
        total_task += float(task_loss.item())
        total_ent  += float(ent.item())

        probs = torch.sigmoid(logits).detach()
        preds = (probs > 0.5).float().detach()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.detach().cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / max(len(loader), 1)
    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["task_loss"] = total_task / max(len(loader), 1)
    metrics["attn_entropy"] = total_ent / max(len(loader), 1)

    return avg_loss, metrics


class EarlyStopping:
    def __init__(self, patience=10, verbose=True, delta=0.0, path="checkpoint.pth"):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_value = np.inf
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_value - self.delta:
            self.best_value = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f"Validation loss decreased → {val_loss:.6f}. Saving model to {self.path}")
        torch.save(model.state_dict(), self.path)

In [ ]:
def pma_output_stats(post_pma_embs: Optional[torch.Tensor]) -> dict:
    if post_pma_embs is None or (not isinstance(post_pma_embs, torch.Tensor)):
        return {"pma_var": 0.0, "pma_std": 0.0}

    x = post_pma_embs.detach()
    if x.dim() != 2 or x.numel() == 0:
        return {"pma_var": 0.0, "pma_std": 0.0}

    var_mean = x.var(dim=0, unbiased=False).mean().item()
    std_mean = x.std(dim=0, unbiased=False).mean().item()

    return {"pma_var": float(var_mean), "pma_std": float(std_mean)}

def attention_entropy(attn: torch.Tensor, pad_mask: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    if attn is None:
        return torch.tensor(0.0, device=pad_mask.device)

    p = attn.masked_fill(pad_mask, 0.0)

    Z = p.sum(dim=-1, keepdim=True).clamp(min=eps)
    p = p / Z

    p_safe = p.clamp(min=eps)
    ent = -(p * torch.log(p_safe)).sum(dim=-1)  
    
    return ent.mean()

In [ ]:
def to_numpy(features):
    if isinstance(features, torch.Tensor):
        return features.detach().cpu().numpy()
    elif isinstance(features, list):
        return np.array(features)
    return features

def plot_and_save_heatmap(matrix, title, save_path, xlabel=None, ylabel=None):
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        matrix,
        cmap='RdBu_r',
        vmin=-1,
        vmax=1,
        center=0,
        xticklabels=False,
        yticklabels=False,
        cbar_kws={'label': 'Cosine Similarity'}
    )
    plt.title(title)
    if xlabel: plt.xlabel(xlabel)
    if ylabel: plt.ylabel(ylabel)
    plt.tight_layout()
    
    plt.savefig(save_path, dpi=300)
    plt.close() 
    print(f"Saved plot to {save_path}")



def analyze_self_similarity(features, sample_idx, save_dir, exp_name, tag_name):
    data = to_numpy(features)
    if data.ndim == 3:
        sample_data = data[sample_idx]
    else:
        sample_data = data 

    sim_matrix = cosine_similarity(sample_data)
    
    title = f'{tag_name} Self-Similarity (Sample {sample_idx})'
    save_path = os.path.join(save_dir, f"{exp_name}_{tag_name}_self_sim_idx{sample_idx}.png")
    
    plot_and_save_heatmap(sim_matrix, title, save_path, 'Sequence index', 'Sequence index')


def analyze_cross_similarity(features, idx_a, idx_b, save_dir, exp_name, tag_name):
    data = to_numpy(features)
    sample_a = data[idx_a]
    sample_b = data[idx_b]

    sim_matrix = cosine_similarity(sample_a, sample_b)
    
    title = f"{tag_name} Cross-Similarity\n(Sample {idx_a} vs Sample {idx_b})"
    save_path = os.path.join(save_dir, f"{exp_name}_{tag_name}_cross_sim_idx{idx_a}_vs_{idx_b}.png")
    
    plot_and_save_heatmap(sim_matrix, title, save_path, f"Sequences of Sample {idx_b}", f"Sequences of Sample {idx_a}")


def analyze_batch_similarity(features, save_dir, exp_name, tag_name):
    data = to_numpy(features)
    
    if data.ndim == 3 and data.shape[1] == 1:
        data = data.squeeze(1)
    elif data.ndim == 3 and data.shape[1] > 1:
        print(f"Warning: {tag_name} shape is {data.shape}. Applying mean pooling for batch similarity.")
        data = data.mean(axis=1)

    sim_matrix = cosine_similarity(data)
    
    title = f'{tag_name} Batch Similarity (N x N)'
    save_path = os.path.join(save_dir, f"{exp_name}_{tag_name}_batch_sim.png")
    
    plot_and_save_heatmap(sim_matrix, title, save_path, 'Sample Index', 'Sample Index')

In [ ]:
@torch.no_grad()
def test(
    model, 
    loader, 
    criterion, 
    device,
    save_dir,
    attn_entropy_lambda: float = 0.0,
    run_name: str = "test_run",
    attn_entropy_mode: str = "min",
    collect_attn: bool = True,
):
    model.eval()

    total_loss = 0.0
    total_task = 0.0
    total_ent  = 0.0

    all_preds, all_labels, all_probs = [], [], []
    attn_rows = []
    
    all_valid_attn_values = [] 
    first_sample_attn_values = []
    global_sample_idx = 0

    GLOBAL_FEATURES = {
        "pre_pma": [],
        "post_pma": [],
        "attn": []
    }

    print(f" Testing [{run_name}]...")

    for X, y, pad_mask in loader:
        X = X.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.float32).view(-1, 1)
        pad_mask = pad_mask.to(device)

        logits, pre_pma_embs, post_pma_embs = model(X, pad_mask)
        logits = logits.float()

        GLOBAL_FEATURES["pre_pma"].append(pre_pma_embs.detach().cpu())
        GLOBAL_FEATURES["post_pma"].append(post_pma_embs.detach().cpu())

        step_attn = getattr(model, "attn_score", None)
        if isinstance(step_attn, torch.Tensor):
            GLOBAL_FEATURES["attn"].append(step_attn.detach().cpu())

        task_loss = criterion(logits, y)
        ent = attention_entropy(step_attn, pad_mask) 

        if attn_entropy_lambda > 0:
            if attn_entropy_mode == "min":
                loss = task_loss + attn_entropy_lambda * ent
            elif attn_entropy_mode == "max":
                loss = task_loss - attn_entropy_lambda * ent
            else:
                loss = task_loss
        else:
            loss = task_loss

        total_loss += loss.item()
        total_task += task_loss.item()
        total_ent  += ent.item()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

        if collect_attn and isinstance(step_attn, torch.Tensor):
            attn_cpu = step_attn.detach().cpu()
            pad_cpu = pad_mask.detach().cpu()

            B, L = attn_cpu.shape
            for i in range(B):
                valid_len = int((~pad_cpu[i]).sum())
                if valid_len > 0:
                    a = attn_cpu[i, :valid_len].numpy().astype(np.float64)
                    a = a / (a.sum() + 1e-12)
    
                    if global_sample_idx == 0:
                        first_sample_attn_values.extend(a.tolist())

                    ent_i = float(-(a * np.log(a + 1e-12)).sum())
                    top1 = float(a.max())
                else:
                    ent_i, top1 = 0.0, 0.0

                attn_rows.append({
                    "sample_idx": global_sample_idx,
                    "true_label": float(y[i].item()),
                    "pred_prob": float(probs[i].item()),
                    "pred_label": float(preds[i].item()),
                    "attn_entropy": ent_i,
                    "attn_top1": top1,
                })
                global_sample_idx += 1

    avg_loss = total_loss / max(len(loader), 1)
    
    metrics = compute_metrics(
        np.array(all_labels).flatten(), 
        np.array(all_preds).flatten(), 
        np.array(all_probs).flatten()
    )
    metrics["task_loss"] = total_task / max(len(loader), 1)
    metrics["attn_entropy"] = total_ent / max(len(loader), 1)

    predictions = {"labels": all_labels, "preds": all_preds, "probs": all_probs}

    all_post_pma = torch.cat(GLOBAL_FEATURES["post_pma"], dim=0)
    pma_stats = pma_output_stats(all_post_pma)
    metrics.update(pma_stats)

    os.makedirs(save_dir, exist_ok=True)
    
    metrics_path = os.path.join(save_dir, "test_metrics.csv")
    pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
    print(f"   --> Saved metrics to {metrics_path}")


    pred_df = pd.DataFrame({
        "label": np.array(all_labels).flatten(), 
        "prob": np.array(all_probs).flatten(),
        "pred": np.array(all_preds).flatten()
    })
    
    pred_path = os.path.join(save_dir, "test_predictions.csv")
    pred_df.to_csv(pred_path, index_label="sample_idx")
    
    try:
        plt.figure(figsize=(8, 6))
        sns.kdeplot(data=pred_df, x="prob", hue="label", fill=True, common_norm=False, palette="coolwarm", alpha=0.3)
        plt.title(f"Prediction Probability Density by Label\n{run_name}")
        plt.xlabel("Predicted Probability (Autism)")
        plt.xlim(0, 1)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "test_prob_density.png"), dpi=300)
        plt.close()
        print(f"   --> Saved probability density plot")
    except Exception as e:
        print(f"   --> Warning: Failed to plot prob density: {e}")

    if collect_attn and attn_rows:
        attn_summary_path = os.path.join(save_dir, "test_attention_summary.csv")
        pd.DataFrame(attn_rows).to_csv(attn_summary_path, index=False)
        print(f"   --> Saved attention summary to {attn_summary_path}")
        
        if len(first_sample_attn_values) > 0:
            try:
                plt.figure(figsize=(8, 6))
                plot_data = first_sample_attn_values
                
                sns.kdeplot(plot_data, fill=True, color="purple")
                plt.title(f"Attention Score Distribution (Sample #0)\n{run_name}")                
                plt.xlabel("Attention Score")
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig(os.path.join(save_dir, "test_attn_density_sample0.png"), dpi=300)
                plt.close()
                print(f"   --> Saved attention score density plot")
            except Exception as e:
                print(f"   --> Warning: Failed to plot attn density: {e}")

    if len(GLOBAL_FEATURES["pre_pma"]) > 0:
        all_pre_pma = torch.cat(GLOBAL_FEATURES["pre_pma"], dim=0)
    else:
        all_pre_pma = None

    if all_post_pma is not None and all_pre_pma is not None:
        try:
            analyze_batch_similarity(
                features=all_post_pma, 
                save_dir=save_dir,
                exp_name=run_name, 
                tag_name="post_pma"
            )
            if all_pre_pma.shape[0] > 0:
                analyze_self_similarity(
                    features=all_pre_pma, 
                    sample_idx=0, 
                    save_dir=save_dir,
                    exp_name=run_name, 
                    tag_name="pre_pma"
                )
            if all_pre_pma.shape[0] > 1:
                analyze_cross_similarity(
                    features=all_pre_pma, 
                    idx_a=0, 
                    idx_b=1, 
                    save_dir=save_dir,
                    exp_name=run_name, 
                    tag_name="pre_pma"
                )
        except Exception as e:
            print(f"   --> Warning: Failed to create heatmaps. Error: {e}")

    attn_df = pd.DataFrame(attn_rows) if collect_attn else None

    print(f" Test Result Summary:")
    print(f"   - AUROC: {metrics['auroc']:.4f}")
    print(f"   - PMA Variance: {metrics['pma_var']:.4f}")
    print(f"   - Avg Entropy: {metrics['attn_entropy']:.4f}")

    return avg_loss, metrics, predictions, attn_df

In [ ]:
def train(
    model,
    train_loader,
    val_loader,
    test_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    num_epochs,
    experiment_name,
    run_name,
    checkpoint_root,
    config_dict=None,
    gradient_clip=1.0,
    group_name=None,
    wandb_project_name="ASD_SET_TRAIN",
    early_patience=5,
    debug=False,
    attn_entropy_lambda: float = 0.0,
    attn_entropy_mode: str = "min",
    projector=None,       
    vicreg_criterion=None,
    vicreg_lambda=0.0     
):
    
    save_dir = os.path.join(checkpoint_root, experiment_name, run_name)
    os.makedirs(save_dir, exist_ok=True)
    
    checkpoint_path = os.path.join(save_dir, "best_model.pth")

    wandb.init(
        project=wandb_project_name,
        name=experiment_name,
        group=group_name,
        config=config_dict or {},
    )

    wandb.config.update({
        "attn_entropy_lambda": float(attn_entropy_lambda),
        "attn_entropy_mode": str(attn_entropy_mode),
        "vicreg_lambda": float(vicreg_lambda),
    }, allow_val_change=True)

    early_stopping = EarlyStopping(patience=early_patience, verbose=True, path=checkpoint_path)

    print(f"\n{'='*80}")
    print(f"Training: {experiment_name}")
    print(f"Checkpoint path: {checkpoint_path}")
    print(f"Early stopping on: val/loss (min) | patience={early_patience}")
    print(f"Debug: {debug}")
    print(f"Entropy reg: lambda={attn_entropy_lambda} | mode={attn_entropy_mode}")
    print(f"VICReg reg: lambda={vicreg_lambda}")
    print(f"{'='*80}\n")

    epoch_bar = tqdm(range(1, num_epochs + 1), desc="Training progress", ncols=110)

    for epoch in epoch_bar:
        train_loss, train_metrics = train_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            epoch=epoch,
            gradient_clip=gradient_clip,
            debug=debug,
            attn_entropy_lambda=attn_entropy_lambda,
            attn_entropy_mode=attn_entropy_mode,
            projector=projector,            
            vicreg_criterion=vicreg_criterion, 
            vicreg_lambda=vicreg_lambda      
        )

        val_loss, val_metrics = validate(
            model, val_loader, criterion, device,
            attn_entropy_lambda=attn_entropy_lambda,
            attn_entropy_mode=attn_entropy_mode,
        )

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        epoch_bar.set_postfix({
            "epoch": epoch,
            "train_loss": f"{train_loss:.4f}",
            "val_loss": f"{val_loss:.4f}",
            "val_auroc": f"{val_metrics['auroc']:.4f}",
            "val_f1": f"{val_metrics['f1']:.4f}",
            "val_ent": f"{val_metrics['attn_entropy']:.3f}",
        })

        log_dict = {
            "epoch": epoch,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/task_loss": float(train_metrics.get("task_loss", np.nan)),
            "train/attn_entropy": float(train_metrics.get("attn_entropy", np.nan)),
            "train/vicreg_loss": float(train_metrics.get("vicreg_loss", np.nan)),
            "val/task_loss": float(val_metrics.get("task_loss", np.nan)),
            "val/attn_entropy": float(val_metrics.get("attn_entropy", np.nan)),
            **{f"train/{k}": v for k, v in train_metrics.items() if k not in ["task_loss", "attn_entropy"]},
            **{f"val/{k}": v for k, v in val_metrics.items() if k not in ["task_loss", "attn_entropy"]},
            "lr": float(optimizer.param_groups[0]["lr"]),
        }

        exclude_keys = ["task_loss", "attn_entropy", "vicreg_loss"]
        log_dict.update({f"train/{k}": v for k, v in train_metrics.items() if k not in exclude_keys})
        log_dict.update({f"val/{k}": v for k, v in val_metrics.items() if k not in exclude_keys})

        wandb.log(log_dict)

        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch}. Best val/loss: {early_stopping.best_value:.6f}")
            break

    print(f"\n{'='*80}")
    print(f"Testing best model (best val/loss: {early_stopping.best_value:.6f})")
    print(f"{'='*80}")

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    test_loss, test_metrics, test_predictions, attn_df = test(
        model, 
        test_loader, 
        criterion, 
        device,
        save_dir=save_dir,
        run_name = experiment_name,
        attn_entropy_lambda=float(attn_entropy_lambda),
        attn_entropy_mode=attn_entropy_mode,
        collect_attn=True,   
    )

    wandb.log({
        "test/loss": test_loss,
        "test/task_loss": float(test_metrics.get("task_loss", np.nan)),
        "test/attn_entropy": float(test_metrics.get("attn_entropy", np.nan)),
        **{f"test/{k}": v for k, v in test_metrics.items() if k not in ["task_loss", "attn_entropy"]},
    })

    print(f"Best model saved at: {checkpoint_path}")
    print(f"Test AUROC: {test_metrics['auroc']:.4f} | Test AUPRC: {test_metrics['auprc']:.4f}")
    print(f"Test attn_entropy(avg, batch-mean): {test_metrics['attn_entropy']:.4f} (lambda={attn_entropy_lambda}, mode={attn_entropy_mode})")

    return test_metrics, test_predictions


In [ ]:
tracker_path = os.path.join(checkpoint_dir, experiment_name, "best_tracker.json")
config_path = os.path.join(checkpoint_dir, experiment_name, "best_config.json")

In [ ]:
sweep_config = {
  "method": "grid",
  "metric": {"name": "val/loss", "goal": "minimize"},
  "parameters": {
    "num_layers": {"values": [2]},
    "num_heads": {"values": [4]},
    "lr": {"values": [1e-5]},
    "dropout": {"values": [0.2]},
    "hidden_dims": {"values": [[256]]},
    "attn_entropy_lambda": {"values": [0.1]},
    "attn_entropy_mode": {"values": ["min"]},
    "temperature": {"values": [0.3]},

    "vicreg_lambda": {
        "values": [0.1] 
    },
    
    "vicreg_proj_dim": {
        "values": [2048]
    },
    
    "vicreg_std_coeff": {
        "values": [50.0]
    },
    "vicreg_sim_coeff": {
        "values": [25.0]
    }
  }
}
sweep_id = wandb.sweep(sweep_config, project=wandb_project_name)
print(f"Sweep created: {sweep_id}")

In [ ]:
@torch.no_grad()
def _collect_test_attention(model, test_loader, device, max_batches=None):
    model.eval()
    attn_list, y_list = [], []

    for b, (X, y, pad_mask) in enumerate(test_loader):
        if (max_batches is not None) and (b >= max_batches):
            break

        X = X.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.float32).view(-1, 1)
        pad_mask = pad_mask.to(device)  

        _ = model(X, pad_mask)

        attn = getattr(model, "attn_score", None)
        attn_cpu = attn.detach().cpu().numpy()                
        pad_cpu  = pad_mask.detach().cpu().numpy().astype(bool) 
        y_cpu    = y.detach().cpu().numpy().reshape(-1).astype(int)

        B, L = attn_cpu.shape
        for i in range(B):
            valid_len = int((~pad_cpu[i]).sum())
            if valid_len <= 0:
                continue
            a = attn_cpu[i, :valid_len].astype(np.float64)

            if np.any(~np.isfinite(a)):
                continue

            attn_list.append(a.astype(np.float32))
            y_list.append(int(y_cpu[i]))

    return attn_list, y_list


def _normalize_prob(a, eps=1e-12):
    a = np.array(a, dtype=np.float64)
    a = np.clip(a, eps, None)
    s = a.sum()
    if s <= 0:
        return None
    return (a / s).astype(np.float64)


def _attn_entropy_np(a, eps=1e-12):
    p = _normalize_prob(a, eps=eps)
    if p is None:
        return 0.0
    return float(-(p * np.log(p + eps)).sum())


def _attn_top1_np(a, eps=1e-12):
    p = _normalize_prob(a, eps=eps)
    if p is None:
        return 0.0
    return float(p.max())


def log_test_attention_histograms(model, test_loader, device, prefix="test", max_batches=None):
    attn_list, y_list = _collect_test_attention(model, test_loader, device, max_batches=max_batches)

    ent  = np.array([_attn_entropy_np(a) for a in attn_list], dtype=float)
    top1 = np.array([_attn_top1_np(a) for a in attn_list], dtype=float)
    y    = np.array(y_list, dtype=int)

    if len(attn_list) > 0:
        a0 = _normalize_prob(attn_list[0])
        L0 = len(attn_list[0])
        print(f"[ATTN DEBUG] n={len(attn_list)} | L0={L0} | first_sum={a0.sum():.4f} | max={a0.max():.4f}")
        print(f"[ATTN DEBUG] entropy_mean={ent.mean():.4f} | top1_mean={top1.mean():.4f} | ln(L0)={np.log(max(L0,1)):.4f}")

    fig1 = plt.figure()
    if (y == 0).any():
        plt.hist(ent[y == 0], bins=40, alpha=0.6, label="label=0")
    if (y == 1).any():
        plt.hist(ent[y == 1], bins=40, alpha=0.6, label="label=1")
    plt.title(f"{prefix} attention entropy")
    plt.xlabel("entropy (max ~ ln(L))")
    plt.ylabel("count")
    plt.legend()
    wandb.log({f"{prefix}/attn_entropy_hist": wandb.Image(fig1)})
    plt.close(fig1)

    fig2 = plt.figure()
    if (y == 0).any():
        plt.hist(top1[y == 0], bins=40, alpha=0.6, label="label=0")
    if (y == 1).any():
        plt.hist(top1[y == 1], bins=40, alpha=0.6, label="label=1")
    plt.title(f"{prefix} attention top1 mass")
    plt.xlabel("top1 mass (in [0,1])")
    plt.ylabel("count")
    plt.legend()
    wandb.log({f"{prefix}/attn_top1_hist": wandb.Image(fig2)})
    plt.close(fig2)

    wandb.summary[f"{prefix}/attn_entropy_mean"] = float(ent.mean()) if len(ent) else 0.0
    wandb.summary[f"{prefix}/attn_top1_mean"] = float(top1.mean()) if len(top1) else 0.0


def sweep_train(config=None):
    with wandb.init(config=config):

        ENCODER_DIM = MODEL_DIM
        p_dim = ENCODER_DIM * 4
        
        config = wandb.config
        v_lambda = config.vicreg_lambda
        std_c    = config.vicreg_std_coeff
        sim_c    = config.vicreg_sim_coeff

        run_name = (
            f"layers{config.num_layers}_"
            f"heads{config.num_heads}_"
            f"lr{config.lr}_"
            f"drop{config.dropout}_"
            f"ent{config.attn_entropy_lambda}_"
            f"temper{config.temperature}_"
            f"vic{v_lambda}_dim{p_dim}_"     
            f"std{std_c:.0f}_sim{sim_c:.0f}"
        )
        wandb.run.name = run_name

        sweep_configs = {
            "set_transformer": {
                "input_dim": MODEL_DIM,
                "num_heads": int(config.num_heads),
                "num_encoder_layers": int(config.num_layers),
                "activation": "gelu",
                "dropout": float(config.dropout),
            },
            "pooling": {
                "input_dim": MODEL_DIM,
                "num_heads": int(config.num_heads),
                "normalize": None,
                "dropout": float(config.dropout),
                "initial_temperature": float(config.temperature)
            },
            "linear": {
                "input_dim": MODEL_DIM,
                "hidden_dims": list(config.hidden_dims),
                "activation": "relu",
                "output_dim": 1,
                "dropout": float(config.dropout),
            }
        }

        model = ASDPredictor(configs_of_modules=sweep_configs, device=str(device)).to(device)

        projector = Projector(
            input_dim=MODEL_DIM, 
            hidden_dim=int(p_dim), 
            output_dim=int(p_dim)  
        ).to(device)

        vicreg_criterion = VICRegLoss(
            sim_coeff=float(sim_c),
            std_coeff=float(std_c),
            cov_coeff=1.0           
        ).to(device)


        train_loader, val_loader, test_loader = prepare_dl(
            datasets=[train_X, val_X, test_X],
            labels=[train_y, val_y, test_y],
            pad_masks=[train_pad, val_pad, test_pad],
            batch_size=int(batch_size),
            dl_seed=42,
            device=device
        )

        y_np = train_y.detach().cpu().numpy().astype(int).reshape(-1)
        pos_num = int((y_np == 1).sum())
        neg_num = int((y_np == 0).sum())
        N = pos_num + neg_num

        w_pos = (N / (2.0 * pos_num)) if pos_num > 0 else 1.0
        w_neg = (N / (2.0 * neg_num)) if neg_num > 0 else 1.0

        criterion = WeightedBCEWithLogitsLoss(w_pos=w_pos, w_neg=w_neg).to(device)


        params = list(model.parameters()) + list(projector.parameters())
        optimizer = optim.AdamW(params, lr=float(config.lr), weight_decay=1e-2)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

        test_metrics, test_predictions = train(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            num_epochs=100,
            
            checkpoint_root=checkpoint_dir, 
            experiment_name=experiment_name,
            run_name=run_name,
            
            config_dict=dict(config),
            gradient_clip=1.0,
            group_name=wandb_project_name,
            wandb_project_name=wandb_project_name,
            debug=True,
            attn_entropy_lambda=float(config.attn_entropy_lambda),
            attn_entropy_mode=str(config.attn_entropy_mode),
            projector=projector,
            vicreg_criterion=vicreg_criterion,
            vicreg_lambda=float(v_lambda)
        )

        current_auroc = test_metrics['auroc']  
        
        tracker_file = os.path.join(checkpoint_dir, experiment_name, "best_tracker.json")
        current_run_dir = os.path.join(checkpoint_dir, experiment_name, run_name)
        
        if os.path.exists(tracker_file):
            try:
                with open(tracker_file, 'r') as f:
                    data = json.load(f)
                    global_best_auroc = data.get("best_auroc", -1.0)
                    previous_best_dir = data.get("run_dir", None)
            except:
                global_best_auroc = -1.0
                previous_best_dir = None
        else:
            global_best_auroc = -1.0
            previous_best_dir = None
            
        print(f"\nComparing AUROC: Current({current_auroc:.4f}) vs Global Best({global_best_auroc:.4f})")

        if current_auroc > global_best_auroc:
            print(f"New Best Run Found! (Higher AUROC) Updating tracker...")
            
            with open(tracker_file, 'w') as f:
                json.dump({"best_auroc": current_auroc, "run_dir": current_run_dir}, f)
            
            with open(os.path.join(checkpoint_dir, experiment_name, "best_config.json"), 'w') as f:
                json.dump(dict(config), f)
            
            if previous_best_dir and os.path.exists(previous_best_dir) and previous_best_dir != current_run_dir:
                try:
                    shutil.rmtree(previous_best_dir)
                    print(f"Deleted previous best run: {previous_best_dir}")
                except Exception as e:
                    print(f"Error deleting previous run: {e}")
                    
            best_ckpt = os.path.join(current_run_dir, "best_model.pth")
            if os.path.exists(best_ckpt):
                model.load_state_dict(torch.load(best_ckpt, map_location=device))
        
        else:
            print(f"Current run({current_auroc:.4f}) is lower than best({global_best_auroc:.4f}). Deleting...")
            if os.path.exists(current_run_dir):
                try:
                    shutil.rmtree(current_run_dir)
                    print(f"Deleted current run: {current_run_dir}")
                except Exception as e:
                    print(f"Error deleting current run: {e}")
        
        log_test_attention_histograms(
            model=model,
            test_loader=test_loader,
            device=device,
            prefix="test",
            max_batches=None
        )

        for k, v in test_metrics.items():
            wandb.summary[f"test/{k}"] = v
            wandb.summary[f"final/test_{k}"] = v

In [ ]:
def run_final_verification(seeds=[42, 43, 44]):
    config_path = os.path.join(checkpoint_dir, experiment_name, "best_config.json")
    if not os.path.exists(config_path):
        print(f" Best config not found at {config_path}. Run sweep first.")
        return

    with open(config_path, 'r') as f:
        config = json.load(f)
    
    print(f"\n Starting Final Verification with Best Config (seeds={seeds})")
    print(f"Loaded Config: {config}")
    
    results = []
    
    for seed_i in seeds:
        print(f"\n{'='*40}")
        print(f"  Running Fixed Seed: {seed_i}")
        print(f"{'='*40}")
        gc.collect()
        torch.cuda.empty_cache()
        
        set_seed(seed_i)
        
        ENCODER_DIM = MODEL_DIM
        p_dim = ENCODER_DIM * 4
        v_lambda = float(config['vicreg_lambda'])
        std_c = float(config['vicreg_std_coeff'])
        sim_c = float(config['vicreg_sim_coeff'])
        
        sweep_configs = {
            "set_transformer": {
                "input_dim": MODEL_DIM,
                "num_heads": int(config['num_heads']),
                "num_encoder_layers": int(config['num_layers']),
                "activation": "gelu",
                "dropout": float(config['dropout']),
            },
            "pooling": {
                "input_dim": MODEL_DIM,
                "num_heads": int(config['num_heads']),
                "normalize": None,
                "dropout": float(config['dropout']),
                "initial_temperature": float(config['temperature'])
            },
            "linear": {
                "input_dim": MODEL_DIM,
                "hidden_dims": list(config['hidden_dims']),
                "activation": "relu",
                "output_dim": 1,
                "dropout": float(config['dropout']),
            }
        }

        model = ASDPredictor(configs_of_modules=sweep_configs, device=str(device)).to(device)
        projector = Projector(input_dim=MODEL_DIM, hidden_dim=int(p_dim), output_dim=int(p_dim)).to(device)
        vicreg_criterion = VICRegLoss(sim_coeff=sim_c, std_coeff=std_c, cov_coeff=1.0).to(device)

        train_loader, val_loader, test_loader = prepare_dl(
            datasets=[train_X, val_X, test_X],
            labels=[train_y, val_y, test_y],
            pad_masks=[train_pad, val_pad, test_pad],
            batch_size=int(batch_size),
            dl_seed=seed_i,   
            device=device
        )
        
        y_np = train_y.detach().cpu().numpy().astype(int).reshape(-1)
        pos_num = int((y_np == 1).sum())
        neg_num = int((y_np == 0).sum())
        N = pos_num + neg_num
        w_pos = (N / (2.0 * pos_num)) if pos_num > 0 else 1.0
        w_neg = (N / (2.0 * neg_num)) if neg_num > 0 else 1.0
        
        criterion = WeightedBCEWithLogitsLoss(w_pos=w_pos, w_neg=w_neg).to(device)
        params = list(model.parameters()) + list(projector.parameters())
        optimizer = optim.AdamW(params, lr=float(config['lr']), weight_decay=1e-2)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

        run_name_seed = f"seed_test/FINAL_BEST_seed{seed_i}"
        
        test_metrics, _ = train(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            num_epochs=100, 
            
            checkpoint_root=checkpoint_dir, 
            experiment_name=experiment_name, 
            run_name=run_name_seed,         
            
            config_dict=config,
            gradient_clip=1.0,
            wandb_project_name=wandb_project_name, 
            debug=False,
            attn_entropy_lambda=float(config['attn_entropy_lambda']),
            attn_entropy_mode=str(config['attn_entropy_mode']),
            projector=projector,
            vicreg_criterion=vicreg_criterion,
            vicreg_lambda=v_lambda
        )
        
        metrics_copy = test_metrics.copy()
        metrics_copy['seed'] = seed_i
        results.append(metrics_copy)
    
    df = pd.DataFrame(results)
    
    save_path_raw = os.path.join(checkpoint_dir, experiment_name, "seed_test/final_3seeds_raw.csv")
    df.to_csv(save_path_raw, index=False)
    

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'seed' in numeric_cols: numeric_cols.remove('seed')
    
    summary = df[numeric_cols].agg(['mean', 'std'])
    save_path_summary = os.path.join(checkpoint_dir, experiment_name, "seed_test/final_3seeds_summary.csv")
    summary.to_csv(save_path_summary)
    
    print(f"\n Final Verification Completed!")
    print(f"   - Raw results: {save_path_raw}")
    print(f"   - Summary (Mean/Std): {save_path_summary}")
    print("\n[Summary Table]")
    print(summary)

run_final_verification()